# # PRE_10 — Ventas minoristas
#
# **Qué se hace:** a partir de `data/sales.csv` (1.000 pedidos de 2022) se revisa la calidad de los
# datos y se generan 9 tablas de apoyo en `submission/`: KPIs, categorías, tipo de día, meses,
# medios de pago, segmentos prioritarios, riesgo de devolución, mejores clientes y mejores productos.
#
# **Definiciones usadas**
# - *Ingreso* = `TotalAmount`. *Ingreso devuelto* = ingreso de pedidos con `IsReturned = 1`.
#   *Ingreso neto* = ingreso - ingreso devuelto.
# - *Tasa de devolución* = pedidos devueltos / pedidos.
# - *Fin de semana* = sábado y domingo.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

# Funciona tanto desde la carpeta de la actividad como desde notebooks/ (Jupyter)
ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
SUBMISSION = ROOT / "submission"
SUBMISSION.mkdir(exist_ok=True)

sales = pd.read_csv(ROOT / "data" / "sales.csv")
print(sales.shape)
sales.head()

(1000, 12)


,OrderID,ProductID,Category,ProductName,Quantity,Price,TotalAmount,OrderDate,CustomerID,PaymentMethod,SalesChannel,IsReturned
0,1,272,Electronics,Product_272,3,54.21,162.63,2022-04-21 01:48:37,26,Credit Card,In-Store,1
1,2,147,Electronics,Product_147,3,163.68,491.04,2022-02-26 03:09:32,38,Cash,Website,1
2,3,217,Fashion,Product_217,5,205.01,1025.05,2022-07-24 07:00:57,271,Cash,Website,0
3,4,292,Electronics,Product_292,4,163.13,652.52,2022-01-09 09:49:05,414,Cash,Website,1
4,5,423,Fashion,Product_423,3,608.73,1826.19,2022-05-19 09:38:52,368,Cash,Mobile App,1


# ## 1. Calidad de datos
# Se revisan nulos, duplicados, coherencia `TotalAmount = Quantity * Price`, fechas y valores válidos.

In [2]:
checks = {
    "filas": len(sales),
    "nulos_totales": int(sales.isna().sum().sum()),
    "filas_duplicadas": int(sales.duplicated().sum()),
    "OrderID_repetidos": int(sales.OrderID.duplicated().sum()),
    "TotalAmount != Quantity*Price (tol. 0.01)": int(((sales.TotalAmount - sales.Quantity * sales.Price).abs() > 0.01).sum()),
    "Quantity <= 0": int((sales.Quantity <= 0).sum()),
    "Price <= 0": int((sales.Price <= 0).sum()),
    "IsReturned fuera de {0,1}": int((~sales.IsReturned.isin([0, 1])).sum()),
}
sales["OrderDate"] = pd.to_datetime(sales["OrderDate"], errors="coerce")
checks["fechas no interpretables"] = int(sales.OrderDate.isna().sum())
print(pd.Series(checks).to_string())
print("Rango de fechas:", sales.OrderDate.min(), "->", sales.OrderDate.max())
# Un ProductID debe tener un único nombre y categoría para poder agrupar por producto
print("ProductID con >1 nombre:", int((sales.groupby("ProductID").ProductName.nunique() > 1).sum()),
      "| con >1 categoría:", int((sales.groupby("ProductID").Category.nunique() > 1).sum()))

filas                                        1000
nulos_totales                                   0
filas_duplicadas                                0
OrderID_repetidos                               0
TotalAmount != Quantity*Price (tol. 0.01)       0
Quantity <= 0                                   0
Price <= 0                                      0
IsReturned fuera de {0,1}                       0
fechas no interpretables                        0
Rango de fechas: 2022-01-01 07:13:02 -> 2022-08-31 23:18:52
ProductID con >1 nombre: 0 | con >1 categoría: 251


# **Conclusión de calidad:** sin nulos, sin duplicados, `TotalAmount` coincide con `Quantity * Price` en
# todas las filas y las fechas se interpretan bien (periodo enero-agosto de 2022, no el año completo).
# No se elimina ni se imputa nada. **Salvedad:** un mismo `ProductID` aparece con varias categorías
# (el dato es sintético), así que la categoría de un producto no es confiable: en `top_products.csv` se
# agrupa por `ProductID` y se informa la categoría más frecuente; los análisis por categoría usan la
# categoría de cada pedido, no la del producto.

In [3]:
sales["month"] = sales.OrderDate.dt.strftime("%Y-%m")
sales["day_type"] = np.where(sales.OrderDate.dt.dayofweek >= 5, "Fin de semana", "Entre semana")
sales["returned_amount"] = sales.TotalAmount * sales.IsReturned
sales["net_amount"] = sales.TotalAmount - sales.returned_amount


def resumen(df, by):
    """Métricas comunes agrupadas por una o varias columnas."""
    g = df.groupby(by).agg(
        orders=("OrderID", "count"),
        units=("Quantity", "sum"),
        revenue=("TotalAmount", "sum"),
        returned_orders=("IsReturned", "sum"),
        returned_revenue=("returned_amount", "sum"),
        net_revenue=("net_amount", "sum"),
    )
    g["avg_ticket"] = g.revenue / g.orders
    g["return_rate"] = g.returned_orders / g.orders
    g["revenue_share"] = g.revenue / df.TotalAmount.sum()
    return g.round({"revenue": 2, "returned_revenue": 2, "net_revenue": 2, "avg_ticket": 2,
                    "return_rate": 4, "revenue_share": 4}).reset_index()


total_rev = sales.TotalAmount.sum()
overall_return = sales.IsReturned.mean()

# ## 2. KPIs globales (`kpi_summary.csv`)

In [4]:
kpi = pd.DataFrame([
    ("orders", len(sales)),
    ("units_sold", sales.Quantity.sum()),
    ("revenue", round(total_rev, 2)),
    ("avg_ticket", round(total_rev / len(sales), 2)),
    ("unique_customers", sales.CustomerID.nunique()),
    ("unique_products", sales.ProductID.nunique()),
    ("orders_per_customer", round(len(sales) / sales.CustomerID.nunique(), 2)),
    ("returned_orders", int(sales.IsReturned.sum())),
    ("return_rate", round(overall_return, 4)),
    ("returned_revenue", round(sales.returned_amount.sum(), 2)),
    ("net_revenue", round(sales.net_amount.sum(), 2)),
    ("first_order_date", sales.OrderDate.min().date().isoformat()),
    ("last_order_date", sales.OrderDate.max().date().isoformat()),
], columns=["metric", "value"])
kpi.to_csv(SUBMISSION / "kpi_summary.csv", index=False)
kpi

,metric,value
0,orders,1000
1,units_sold,3069
2,revenue,1280745.55
3,avg_ticket,1280.75
4,unique_customers,428
5,unique_products,371
6,orders_per_customer,2.34
7,returned_orders,516
8,return_rate,0.516
9,returned_revenue,643703.02


# ## 3. Categoría, tipo de día, mes y medio de pago

In [5]:
category = resumen(sales, "Category").sort_values("revenue", ascending=False)
category.to_csv(SUBMISSION / "category_summary.csv", index=False)

# Ingreso promedio diario: se divide por los días calendario de cada tipo dentro del periodo
days = pd.date_range(sales.OrderDate.min().normalize(), sales.OrderDate.max().normalize())
n_days = pd.Series(np.where(days.dayofweek >= 5, "Fin de semana", "Entre semana")).value_counts()
day_type = resumen(sales, "day_type")
day_type["calendar_days"] = day_type.day_type.map(n_days)
day_type["avg_daily_revenue"] = (day_type.revenue / day_type.calendar_days).round(2)
day_type.to_csv(SUBMISSION / "day_type_summary.csv", index=False)

monthly = resumen(sales, "month").sort_values("month")
monthly["revenue_mom_change"] = monthly.revenue.pct_change().round(4)
monthly.drop(columns="revenue_share").to_csv(SUBMISSION / "monthly_sales.csv", index=False)

payment = resumen(sales, "PaymentMethod").sort_values("revenue", ascending=False)
payment.to_csv(SUBMISSION / "payment_summary.csv", index=False)

display(category, day_type, monthly, payment)

,Category,orders,units,revenue,returned_orders,returned_revenue,net_revenue,avg_ticket,return_rate,revenue_share
1,Fashion,343,1037,451214.50,179,225784.37,225430.13,1315.49,0.5219,0.3523
0,Electronics,336,1043,440470.24,172,226316.49,214153.75,1310.92,0.5119,0.3439
2,Home & Garden,321,989,389060.81,165,191602.16,197458.65,1212.03,0.5140,0.3038


,day_type,orders,units,revenue,returned_orders,returned_revenue,net_revenue,avg_ticket,return_rate,revenue_share,calendar_days,avg_daily_revenue
0,Entre semana,729,2230,937081.93,379,471986.95,465094.98,1285.43,0.5199,0.7317,173,5416.66
1,Fin de semana,271,839,343663.62,137,171716.07,171947.55,1268.13,0.5055,0.2683,70,4909.48


,month,orders,units,revenue,returned_orders,returned_revenue,net_revenue,avg_ticket,return_rate,revenue_share,revenue_mom_change
0,2022-01,109,337,148255.42,48,62458.04,85797.38,1360.14,0.4404,0.1158,NaN
1,2022-02,120,374,138757.64,65,76669.05,62088.59,1156.31,0.5417,0.1083,-0.0641
2,2022-03,131,387,163936.57,68,82569.38,81367.19,1251.42,0.5191,0.1280,0.1815
3,2022-04,126,393,165658.58,74,90280.48,75378.10,1314.75,0.5873,0.1293,0.0105
4,2022-05,117,370,163543.63,57,72929.16,90614.47,1397.81,0.4872,0.1277,-0.0128
5,2022-06,129,393,154568.39,63,76264.10,78304.29,1198.20,0.4884,0.1207,-0.0549
6,2022-07,130,399,171321.77,69,86242.25,85079.52,1317.86,0.5308,0.1338,0.1084
7,2022-08,138,416,174703.55,72,96290.56,78412.99,1265.97,0.5217,0.1364,0.0197


,PaymentMethod,orders,units,revenue,returned_orders,returned_revenue,net_revenue,avg_ticket,return_rate,revenue_share
2,PayPal,346,1050,450827.44,170,223727.53,227099.91,1302.97,0.4913,0.3520
0,Cash,311,984,418676.17,176,234872.46,183803.71,1346.23,0.5659,0.3269
1,Credit Card,343,1035,411241.94,170,185103.03,226138.91,1198.96,0.4956,0.3211


# ## 4. Riesgo de devolución (`return_risk.csv`)
# Tasa de devolución por dimensión (categoría, canal, pago, tipo de día, rango de precio y cantidad).
# `lift` = tasa del grupo / tasa global: > 1 indica más riesgo que el promedio.

In [6]:
sales["price_band"] = pd.qcut(sales.Price, 4, labels=["Q1 bajo", "Q2", "Q3", "Q4 alto"]).astype(str)
sales["quantity_label"] = sales.Quantity.astype(str)
dims = {"Category": "Category", "SalesChannel": "SalesChannel", "PaymentMethod": "PaymentMethod",
        "day_type": "day_type", "price_band": "price_band", "Quantity": "quantity_label"}
parts = []
for dim, col in dims.items():
    g = sales.groupby(col).agg(orders=("OrderID", "count"), returned_orders=("IsReturned", "sum"),
                               revenue=("TotalAmount", "sum"), returned_revenue=("returned_amount", "sum"))
    g.insert(0, "value", g.index)
    g.insert(0, "dimension", dim)
    parts.append(g.reset_index(drop=True))
risk = pd.concat(parts, ignore_index=True)
risk["return_rate"] = (risk.returned_orders / risk.orders).round(4)
risk["lift"] = (risk.return_rate / overall_return).round(3)
# con ~1.000 pedidos, diferencias de pocos puntos pueden ser ruido: se marca solo > 10 % sobre el promedio
risk["risk_level"] = np.where(risk.lift >= 1.10, "alto", np.where(risk.lift <= 0.90, "bajo", "normal"))
risk[["revenue", "returned_revenue"]] = risk[["revenue", "returned_revenue"]].round(2)
risk = risk.sort_values("lift", ascending=False).reset_index(drop=True)
risk.to_csv(SUBMISSION / "return_risk.csv", index=False)
risk.head(10)

,dimension,value,orders,returned_orders,revenue,returned_revenue,return_rate,lift,risk_level
0,PaymentMethod,Cash,311,176,418676.17,234872.46,0.5659,1.097,normal
1,Quantity,1,178,97,72198.12,39316.91,0.5449,1.056,normal
2,SalesChannel,Website,323,175,423478.55,221554.12,0.5418,1.050,normal
3,Quantity,2,195,104,161219.80,89436.98,0.5333,1.034,normal
4,SalesChannel,Mobile App,338,179,435831.09,240448.37,0.5296,1.026,normal
5,price_band,Q1 bajo,250,132,93070.62,49948.49,0.5280,1.023,normal
6,price_band,Q2,250,131,247558.56,125398.98,0.5240,1.016,normal
7,Category,Fashion,343,179,451214.50,225784.37,0.5219,1.011,normal
8,day_type,Entre semana,729,379,937081.93,471986.95,0.5199,1.008,normal
9,Quantity,5,197,102,403491.15,200992.25,0.5178,1.003,normal


# ## 5. Segmentos prioritarios (`priority_segments.csv`)
# Segmento = categoría x canal de venta (9 segmentos). Prioridad **alta** si el segmento vende por encima
# de la mediana y además devuelve más que el promedio global (mucho ingreso en riesgo); **media** si
# cumple solo una condición; **baja** si ninguna.

In [7]:
seg = resumen(sales, ["Category", "SalesChannel"])
big = seg.revenue >= seg.revenue.median()
risky = seg.return_rate > overall_return
seg["priority"] = np.select([big & risky, big | risky], ["alta", "media"], default="baja")
order = {"alta": 0, "media": 1, "baja": 2}
seg = seg.sort_values(["priority", "returned_revenue"], key=lambda s: s.map(order) if s.name == "priority" else -s)
seg = seg.reset_index(drop=True)
seg.to_csv(SUBMISSION / "priority_segments.csv", index=False)
seg

,Category,SalesChannel,orders,units,revenue,returned_orders,returned_revenue,net_revenue,avg_ticket,return_rate,revenue_share,priority
0,Fashion,Mobile App,131,397,167679.44,68,87672.87,80006.57,1280.00,0.5191,0.1309,alta
1,Fashion,Website,112,344,161321.82,61,83604.72,77717.10,1440.37,0.5446,0.1260,alta
2,Electronics,Mobile App,108,323,152164.20,54,78806.27,73357.93,1408.93,0.5000,0.1188,media
3,Electronics,In-Store,121,383,158549.84,59,74284.36,84265.48,1310.33,0.4876,0.1238,media
4,Home & Garden,Mobile App,99,315,115987.45,57,73969.23,42018.22,1171.59,0.5758,0.0906,media
5,Electronics,Website,107,337,129756.20,59,73225.86,56530.34,1212.67,0.5514,0.1013,media
6,Home & Garden,Website,104,323,132400.53,55,64723.54,67676.99,1273.08,0.5288,0.1034,media
7,Home & Garden,In-Store,118,351,140672.83,53,52909.39,87763.44,1192.14,0.4492,0.1098,media
8,Fashion,In-Store,100,296,122213.24,50,54506.78,67706.46,1222.13,0.5000,0.0954,baja


# ## 6. Mejores clientes y productos (top 20 por ingreso)

In [8]:
cust = sales.groupby("CustomerID").agg(
    orders=("OrderID", "count"), units=("Quantity", "sum"), revenue=("TotalAmount", "sum"),
    returned_orders=("IsReturned", "sum"), returned_revenue=("returned_amount", "sum"),
    first_order=("OrderDate", "min"), last_order=("OrderDate", "max"),
)
cust["avg_ticket"] = cust.revenue / cust.orders
cust["return_rate"] = cust.returned_orders / cust.orders
cust["revenue_share"] = cust.revenue / total_rev
cust[["first_order", "last_order"]] = cust[["first_order", "last_order"]].apply(lambda c: c.dt.date.astype(str))
top_customers = (cust.round({"revenue": 2, "returned_revenue": 2, "avg_ticket": 2, "return_rate": 4, "revenue_share": 4})
                 .sort_values("revenue", ascending=False).head(20).reset_index())
top_customers.insert(0, "rank", range(1, len(top_customers) + 1))
top_customers.to_csv(SUBMISSION / "top_customers.csv", index=False)

prod = sales.groupby(["ProductID", "ProductName"]).agg(
    Category=("Category", lambda s: s.mode().iloc[0]),
    orders=("OrderID", "count"), units=("Quantity", "sum"), revenue=("TotalAmount", "sum"),
    returned_orders=("IsReturned", "sum"), returned_revenue=("returned_amount", "sum"),
)
prod["return_rate"] = prod.returned_orders / prod.orders
prod["revenue_share"] = prod.revenue / total_rev
top_products = (prod.round({"revenue": 2, "returned_revenue": 2, "return_rate": 4, "revenue_share": 4})
                .sort_values("revenue", ascending=False).head(20).reset_index())
top_products.insert(0, "rank", range(1, len(top_products) + 1))
top_products.to_csv(SUBMISSION / "top_products.csv", index=False)
display(top_customers.head(5), top_products.head(5))

,rank,CustomerID,orders,units,revenue,returned_orders,returned_revenue,first_order,last_order,avg_ticket,return_rate,revenue_share
0,1,383,4,18,12157.31,1,3428.45,2022-03-01,2022-08-11,3039.33,0.25,0.0095
1,2,157,5,17,11836.95,3,5532.68,2022-03-23,2022-08-10,2367.39,0.60,0.0092
2,3,466,8,27,10623.11,4,2614.20,2022-01-17,2022-07-28,1327.89,0.50,0.0083
3,4,141,4,19,10253.65,1,2824.60,2022-01-04,2022-08-26,2563.41,0.25,0.0080
4,5,160,5,20,9912.24,2,3487.00,2022-01-25,2022-08-24,1982.45,0.40,0.0077


,rank,ProductID,ProductName,Category,orders,units,revenue,returned_orders,returned_revenue,return_rate,revenue_share
0,1,309,Product_309,Electronics,7,27,13879.47,2,4276.55,0.2857,0.0108
1,2,297,Product_297,Electronics,10,32,13531.34,6,7766.31,0.6000,0.0106
2,3,147,Product_147,Home & Garden,5,20,12533.59,2,1920.04,0.4000,0.0098
3,4,390,Product_390,Fashion,7,24,11816.12,5,8239.32,0.7143,0.0092
4,5,193,Product_193,Electronics,6,20,11493.08,4,8216.32,0.6667,0.0090


# ## 7. Verificación de consistencia entre archivos

In [9]:
tol = 0.05
assert abs(category.revenue.sum() - total_rev) < tol
assert abs(payment.revenue.sum() - total_rev) < tol
assert abs(day_type.revenue.sum() - total_rev) < tol
assert abs(monthly.revenue.sum() - total_rev) < tol
assert abs(seg.revenue.sum() - total_rev) < tol
assert category.orders.sum() == payment.orders.sum() == day_type.orders.sum() == monthly.orders.sum() == len(sales)
assert abs(category.net_revenue.sum() + category.returned_revenue.sum() - total_rev) < tol
assert (top_customers.revenue.diff().dropna() <= 0).all() and (top_products.revenue.diff().dropna() <= 0).all()
for name in ["category_summary", "day_type_summary", "kpi_summary", "monthly_sales", "payment_summary",
             "priority_segments", "return_risk", "top_customers", "top_products"]:
    print(f"{name}.csv", len(pd.read_csv(SUBMISSION / f"{name}.csv")), "filas")

category_summary.csv 3 filas
day_type_summary.csv 2 filas
kpi_summary.csv 13 filas
monthly_sales.csv 8 filas
payment_summary.csv 3 filas
priority_segments.csv 9 filas
return_risk.csv 20 filas
top_customers.csv 20 filas
top_products.csv 20 filas


# ## Hallazgos

In [10]:
best_cat = category.iloc[0]
best_month = monthly.loc[monthly.revenue.idxmax()]
print(f"Ingreso total {total_rev:,.0f} | neto {sales.net_amount.sum():,.0f} | devolución {overall_return:.1%}")
print(f"Categoría líder: {best_cat.Category} ({best_cat.revenue_share:.1%} del ingreso)")
print(f"Mejor mes: {best_month.month} ({best_month.revenue:,.0f})")
print("Ticket promedio por tipo de día:", day_type.set_index("day_type").avg_ticket.to_dict())
print("Ingreso diario promedio:", day_type.set_index("day_type").avg_daily_revenue.to_dict())
print("Top 20 clientes concentran:", f"{top_customers.revenue_share.sum():.1%}", "del ingreso")
print("Segmentos de prioridad alta:", seg[seg.priority == "alta"][["Category", "SalesChannel"]].values.tolist())

Ingreso total 1,280,746 | neto 637,043 | devolución 51.6%
Categoría líder: Fashion (35.2% del ingreso)
Mejor mes: 2022-08 (174,704)
Ticket promedio por tipo de día: {'Entre semana': 1285.43, 'Fin de semana': 1268.13}
Ingreso diario promedio: {'Entre semana': 5416.66, 'Fin de semana': 4909.48}
Top 20 clientes concentran: 13.8% del ingreso
Segmentos de prioridad alta: [['Fashion', 'Mobile App'], ['Fashion', 'Website']]


# - El 51,6 % de los pedidos se devuelve y ~50 % del ingreso bruto (1,28 M) corresponde a pedidos devueltos:
#   el **ingreso neto** (0,64 M) es la cifra relevante, no el bruto.
# - Fashion lidera con 35 % del ingreso; agosto es el mejor mes; entre semana se vende más por día
#   (5.417 vs 4.909) pero el ticket promedio es casi igual.
# - La tasa de devolución es alta y bastante pareja entre dimensiones; con solo 1.000 pedidos, las
#   diferencias pequeñas entre grupos no son concluyentes (por eso `risk_level` usa un umbral de 10 %).
# - Los segmentos de prioridad alta (Fashion en App móvil y en Web) son donde conviene actuar primero: venden mucho y devuelven más que la media.
# - Los clientes están poco concentrados (los 20 mejores suman 13,8 % del ingreso), no hay dependencia de uno solo.